In [1]:
path='/content/drive/MyDrive/Research_kgTxagent/Kg_dev_files_and_data/test_openfda/OpenFDA_graph.pkl'


## Installation of LLama3

In [2]:
!apt-get update
!apt-get install -y zstd


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!nohup ollama serve > ollama.log 2>&1 &


In [5]:
import time
time.sleep(5)
!which ollama


/usr/local/bin/ollama


In [6]:

!pip install ollama
!ollama pull llama3



In [9]:
!ollama list

NAME             ID              SIZE      MODIFIED           
llama3:latest    365c0bd3c000    4.7 GB    About a minute ago    


# Install embedder

In [10]:
!ollama pull nomic-embed-text

In [11]:
!curl http://localhost:11434/api/embeddings -d '{"model": "nomic-embed-text", "prompt": "test"}'

{"embedding":[0.6649931073188782,0.2698410749435425,-4.428001880645752,-0.2083042860031128,1.4541720151901245,0.1412624567747116,1.1023385524749756,-0.09349896013736725,0.8581572771072388,-0.6412907242774963,0.11737719178199768,1.5755916833877563,0.7947250008583069,0.13836398720741272,-1.0482560396194458,-1.2551968097686768,1.2373385429382324,-2.092716693878174,-0.1399470716714859,-0.1324055790901184,0.06658345460891724,-0.27080878615379333,-1.3459391593933105,0.16992934048175812,3.1240322589874268,-0.37583813071250916,-1.4518086910247803,1.3718993663787842,-0.6352297067642212,-0.4408784508705139,0.8749979734420776,-0.6551274061203003,1.1642423868179321,-1.789885401725769,-0.7300690412521362,-0.6582493782043457,0.5724857449531555,1.1555099487304688,0.26875966787338257,-0.18783614039421082,-0.20474441349506378,0.6469362378120422,-0.13082054257392883,-0.8160953521728516,1.6506396532058716,0.5055292844772339,0.5379217863082886,1.7876688241958618,0.6332682371139526,-0.8675547242164612,-1.3

In [12]:
!ollama pull nomic-embed-text

In [14]:
!ollama list

NAME                       ID              SIZE      MODIFIED       
nomic-embed-text:latest    0a109f422b47    274 MB    53 seconds ago    
llama3:latest              365c0bd3c000    4.7 GB    4 minutes ago     


# Task
Load a graph from a specified path, generate node embeddings using Ollama's 'nomic-embed-text' model, and store these embeddings as node attributes within the NetworkX graph. The graph is located at '/content/drive/MyDrive/Research_kgTxagent/Kg_dev_files_and_data/test_openfda/OpenFDA_graph.pkl'.

## Load and Normalize Graph with NetworkX

### Subtask:
Load the graph from the specified path ('/content/drive/MyDrive/Research_kgTxagent/Kg_dev_files_and_data/test_openfda/OpenFDA_graph.pkl') into a NetworkX graph object. Perform any necessary normalization or preprocessing steps using NetworkX functionalities.


**Reasoning**:
I need to import the necessary libraries (`networkx` and `pickle`), load the graph from the given path, and then print the number of nodes and edges to confirm it loaded correctly.



In [15]:
import networkx as nx
import pickle

# Load the graph from the specified path
with open(path, 'rb') as f:
    graph = pickle.load(f)

# Print the number of nodes and edges to verify
print(f"Number of nodes in the graph: {graph.number_of_nodes()}")
print(f"Number of edges in the graph: {graph.number_of_edges()}")

Number of nodes in the graph: 110843
Number of edges in the graph: 355909


# understanding graph Schema

In [16]:
import networkx as nx
from collections import Counter

# --- Basic size ---
print(f"Nodes: {graph.number_of_nodes()}")
print(f"Edges: {graph.number_of_edges()}")
print(f"Directed: {graph.is_directed()}")
print(f"Multigraph: {graph.is_multigraph()}")

# --- Sample a few nodes with their full attribute dicts ---
print("\n--- Sample nodes ---")
for i, (node_id, data) in enumerate(graph.nodes(data=True)):
    print(f"Node ID: {node_id!r}")
    print(f"  Attributes: {data}")
    if i >= 4:
        break

# --- Sample a few edges with their full attribute dicts ---
print("\n--- Sample edges ---")
for i, (u, v, data) in enumerate(graph.edges(data=True)):
    print(f"Edge: {u!r} -> {v!r}")
    print(f"  Attributes: {data}")
    if i >= 4:
        break

# --- All unique attribute keys used across nodes ---
node_keys = Counter()
for _, data in graph.nodes(data=True):
    node_keys.update(data.keys())
print("\n--- Node attribute keys (with frequency) ---")
for key, count in node_keys.most_common():
    print(f"  {key}: appears on {count}/{graph.number_of_nodes()} nodes")

# --- All unique attribute keys used across edges ---
edge_keys = Counter()
for _, _, data in graph.edges(data=True):
    edge_keys.update(data.keys())
print("\n--- Edge attribute keys (with frequency) ---")
for key, count in edge_keys.most_common():
    print(f"  {key}: appears on {count}/{graph.number_of_edges()} edges")

# --- If nodes have a 'type' or 'label' field, show the distribution ---
for candidate in ["type", "label", "category", "node_type"]:
    if candidate in node_keys:
        type_counts = Counter(data.get(candidate) for _, data in graph.nodes(data=True))
        print(f"\n--- Node '{candidate}' distribution ---")
        for t, c in type_counts.most_common():
            print(f"  {t}: {c}")

# --- If edges have a 'type' or 'relation' field, show the distribution ---
for candidate in ["type", "relation", "relationship", "edge_type"]:
    if candidate in edge_keys:
        type_counts = Counter(data.get(candidate) for _, _, data in graph.edges(data=True))
        print(f"\n--- Edge '{candidate}' distribution ---")
        for t, c in type_counts.most_common():
            print(f"  {t}: {c}")

# --- Node degree distribution summary ---
degrees = [d for _, d in graph.degree()]
print(f"\n--- Degree stats ---")
print(f"  min: {min(degrees)}, max: {max(degrees)}, avg: {sum(degrees)/len(degrees):.2f}")

Nodes: 110843
Edges: 355909
Directed: False
Multigraph: False

--- Sample nodes ---
Node ID: 'DRUG_GENERIC:6e49f2b49e93d317'
  Attributes: {'id': 'DRUG_GENERIC:6e49f2b49e93d317', 'name': 'LIDOCAINE', 'type': 'drug_generic', 'source': 'FDA_tool_list', 'description': '', 'source_db': '', 'retrieval_date': ''}
Node ID: 'DRUG_GENERIC:293e95c8a06eaa59'
  Attributes: {'id': 'DRUG_GENERIC:293e95c8a06eaa59', 'name': 'SALICYLIC ACID', 'type': 'drug_generic', 'source': 'FDA_tool_list', 'description': '', 'source_db': '', 'retrieval_date': ''}
Node ID: 'DRUG_GENERIC:f775126c48cf5596'
  Attributes: {'id': 'DRUG_GENERIC:f775126c48cf5596', 'name': 'VIBEGRON', 'type': 'drug_generic', 'source': 'FDA_tool_list', 'description': '', 'source_db': '', 'retrieval_date': ''}
Node ID: 'DRUG_GENERIC:3840b059b5d2f0a2'
  Attributes: {'id': 'DRUG_GENERIC:3840b059b5d2f0a2', 'name': 'BENZALKONIUM CHLORIDE, LIDOCAINE', 'type': 'drug_generic', 'source': 'FDA_tool_list', 'description': '', 'source_db': '', 'retrieval_

In [17]:
import numpy as np

# 1. Check existing embeddings: dimension + whether they're plain lists/np arrays
sample_with_emb = next((d for _, d in graph.nodes(data=True) if d.get('embedding') is not None), None)
if sample_with_emb:
    emb = sample_with_emb['embedding']
    print(f"Existing embedding type: {type(emb)}")
    print(f"Existing embedding length/dim: {len(emb) if hasattr(emb, '__len__') else 'N/A'}")
    print(f"Node this came from: {sample_with_emb['name']} ({sample_with_emb['type']})")

# 2. Check what NULL-embedding nodes look like (are they a specific type?)
from collections import Counter
null_emb_types = Counter(d['type'] for _, d in graph.nodes(data=True) if d.get('embedding') is None)
print(f"\nTypes with null embedding: {null_emb_types}")

# 3. Check actual text content for a 'text' type node (not drug_brand/generic)
for node_id, d in graph.nodes(data=True):
    if d['type'] == 'indications_and_usage':
        print(f"\nSample indications_and_usage node:")
        print(f"  name: {d['name'][:300]!r}")
        print(f"  description: {d['description']!r}")
        break

for node_id, d in graph.nodes(data=True):
    if d['type'] == 'warnings':
        print(f"\nSample warnings node:")
        print(f"  name: {d['name'][:300]!r}")
        break


Types with null embedding: Counter({'drug_brand': 31797, 'drug_generic': 10840, 'dosage_and_administration': 9443, 'indications_and_usage': 9442, 'warnings': 7590, 'inactive_ingredient': 7107, 'active_ingredient': 7008, 'keep_out_of_reach_of_children': 6985, 'purpose': 6871, 'stop_use': 2814, 'storage_and_handling': 2688, 'pregnancy_or_breast_feeding': 2388, 'do_not_use': 2082, 'ask_doctor': 1585, 'dosage_and_administration_table': 1454, 'ask_doctor_or_pharmacist': 749})

Sample indications_and_usage node:
  name: 'indications_and_usage ((CALCIUM, MAGNESIUM, POTASSIUM, AND SODIUM OXYBATES))'
  description: '1 INDICATIONS AND USAGE XYWAV is a central nervous system depressant indicated for the treatment of: • Cataplexy or excessive daytime sleepiness (EDS) in patients 7 years of age and older with narcolepsy ( 1.1 ). • Idiopathic Hypersomnia (IH) in adults ( 1.2 ). 1.1 Narcolepsy XYWAV is indicated for the treatment of cataplexy or excessive daytime sleepiness (EDS) in patients 7 years

In [18]:
from collections import Counter, defaultdict

# --- 1. Overall edge type distribution ---
edge_types = Counter()
for u, v, data in graph.edges(data=True):
    edge_types[data.get('relation', data.get('type', 'UNKNOWN'))] += 1

print("--- Edge type distribution ---")
for etype, count in edge_types.most_common():
    print(f"  {etype}: {count}")

# --- 2. All attribute keys used on edges, with frequency ---
edge_keys = Counter()
for _, _, data in graph.edges(data=True):
    edge_keys.update(data.keys())
print("\n--- Edge attribute keys ---")
for key, count in edge_keys.most_common():
    print(f"  {key}: appears on {count}/{graph.number_of_edges()} edges")

# --- 3. Sample a full edge per relation type (not just the first edge overall) ---
seen_types = set()
print("\n--- One sample edge per relation type ---")
for u, v, data in graph.edges(data=True):
    etype = data.get('relation', data.get('type', 'UNKNOWN'))
    if etype not in seen_types:
        seen_types.add(etype)
        u_type = graph.nodes[u].get('type', '?')
        v_type = graph.nodes[v].get('type', '?')
        print(f"\nRelation: {etype}")
        print(f"  {u!r} [{u_type}]  -->  {v!r} [{v_type}]")
        print(f"  Edge data: {data}")

# --- 4. Which node-type pairs connect to which node-type pairs? ---
# This tells us: does a drug_generic connect DIRECTLY to warnings/dosage/etc,
# or is there an intermediate node?
type_pair_counts = Counter()
for u, v, data in graph.edges(data=True):
    u_type = graph.nodes[u].get('type', '?')
    v_type = graph.nodes[v].get('type', '?')
    type_pair_counts[(u_type, v_type)] += 1

print("\n--- Node-type-pair connection patterns (source -> target) ---")
for (ut, vt), count in type_pair_counts.most_common():
    print(f"  {ut} -> {vt}: {count}")

# --- 5. For one specific drug, print its full 1-hop neighborhood ---
# Pick any drug_generic node to see exactly what's attached to it
sample_drug_id = next(nid for nid, d in graph.nodes(data=True) if d.get('type') == 'drug_generic')
sample_drug_data = graph.nodes[sample_drug_id]
print(f"\n--- Full 1-hop neighborhood of sample drug_generic node ---")
print(f"Node: {sample_drug_id!r} -> {sample_drug_data.get('name')}")

for neighbor in graph.neighbors(sample_drug_id):
    n_type = graph.nodes[neighbor].get('type', '?')
    n_name = graph.nodes[neighbor].get('name', '')[:60]
    edge_data = graph.get_edge_data(sample_drug_id, neighbor)
    print(f"  --> [{n_type}] {n_name}  | edge: {edge_data}")

# If directed and neighbors() only shows outgoing, also check incoming
if graph.is_directed():
    print(f"\n--- Incoming edges to same node (if directed graph) ---")
    for predecessor in graph.predecessors(sample_drug_id):
        p_type = graph.nodes[predecessor].get('type', '?')
        p_name = graph.nodes[predecessor].get('name', '')[:60]
        edge_data = graph.get_edge_data(predecessor, sample_drug_id)
        print(f"  <-- [{p_type}] {p_name}  | edge: {edge_data}")

--- Edge type distribution ---
  BRAND_GENERIC: 34877
  HAS_DOSAGE: 9443
  HAS_INDICATION_TEXT: 9442
  DOSAGE_INDICATION: 9415
  HAS_INACTIVE_INGREDIENT_TEXT: 7107
  INACTIVE_INDICATION: 7086
  DOSAGE_INACTIVE: 7081
  HAS_ACTIVE_INGREDIENT_TEXT: 7008
  ACTIVE_INACTIVE: 7006
  ACTIVE_INDICATION: 6990
  KEEP_OUT_OF_REACH: 6985
  ACTIVE_DOSAGE: 6984
  INACTIVE_KEEPCHILDREN: 6981
  INDICATION_KEEPCHILDREN: 6970
  DOSAGE_KEEPCHILDREN: 6961
  ACTIVE_KEEPCHILDREN: 6912
  HAS_PURPOSE: 6871
  INACTIVE_PURPOSE: 6868
  INDICATION_PURPOSE: 6855
  DOSAGE_PURPOSE: 6850
  KEEPCHILDREN_PURPOSE: 6808
  ACTIVE_PURPOSE: 6802
  STOP_USE: 2814
  INACTIVE_STOPUSE: 2814
  INDICATION_STOPUSE: 2808
  KEEPCHILDREN_STOPUSE: 2808
  DOSAGE_STOPUSE: 2805
  ACTIVE_STOPUSE: 2795
  HAS_STORAGE: 2688
  DOSAGE_STORAGE: 2677
  INDICATION_STORAGE: 2676
  PURPOSE_STOPUSE: 2671
  PREGNANCY_LACTATION: 2388
  INACTIVE_PREGNANCY: 2388
  DOSAGE_PREGNANCY: 2386
  INDICATION_PREGNANCY: 2386
  ACTIVE_PREGNANCY: 2385
  KEEPCHILDREN

In [19]:
print(f"Directed: {graph.is_directed()}")
print(f"Multigraph: {graph.is_multigraph()}")

# If directed, confirm predecessors() actually resolves a section back to its drug_generic
if graph.is_directed():
    # grab a warnings-type node
    sample_section = next(nid for nid, d in graph.nodes(data=True) if d.get('type') == 'warnings')
    preds = list(graph.predecessors(sample_section))
    print(f"\nSample section node: {sample_section}")
    print(f"Predecessors ({len(preds)}):")
    for p in preds:
        p_type = graph.nodes[p].get('type', '?')
        p_name = graph.nodes[p].get('name', '')[:50]
        edge_data = graph.get_edge_data(p, sample_section)
        print(f"  <-- [{p_type}] {p_name} | edge: {edge_data}")

    # also confirm neighbors() (successors) is empty or different — sanity check
    succs = list(graph.neighbors(sample_section))
    print(f"\nSuccessors/neighbors of same node ({len(succs)}):")
    for s in succs[:5]:
        print(f"  --> [{graph.nodes[s].get('type','?')}] {graph.nodes[s].get('name','')[:50]}")
else:
    print("\nUndirected — neighbors() already returns both directions, no predecessors() needed.")

Directed: False
Multigraph: False

Undirected — neighbors() already returns both directions, no predecessors() needed.


# Context, Reasoning, and Truth: Auditing Hallucination in LLM

In [20]:
def get_drug_context(matched_node_id: str, graph) -> dict:
    """
    Given a node ID from FAISS search, resolve it to its parent
    drug_generic and return the full label context.
    """
    node_data = graph.nodes[matched_node_id]
    node_type = node_data.get('type')

    if node_type == 'drug_generic':
        generic_id = matched_node_id
    elif node_type in ('drug_brand',) or node_type != 'drug_generic':
        # brand node OR a section node — both need to find their
        # drug_generic neighbor specifically
        generic_candidates = [
            n for n in graph.neighbors(matched_node_id)
            if graph.nodes[n].get('type') == 'drug_generic'
        ]
        if not generic_candidates:
            raise ValueError(f"No drug_generic found for node {matched_node_id} (type={node_type})")
        generic_id = generic_candidates[0]  # should be exactly one

    generic_data = graph.nodes[generic_id]

    # Gather all section neighbors (exclude drug_brand and drug_generic
    # themselves to avoid pulling the redundant mesh or brand list here)
    sections = {}
    brands = []
    for neighbor_id in graph.neighbors(generic_id):
        n_data = graph.nodes[neighbor_id]
        n_type = n_data.get('type')
        if n_type == 'drug_brand':
            brands.append(n_data.get('name'))
        elif n_type != 'drug_generic':
            # section node — use description if populated, else name
            text = n_data.get('description') or n_data.get('name', '')
            sections[n_type] = text

    return {
        'generic_id': generic_id,
        'generic_name': generic_data.get('name'),
        'brands': brands,
        'sections': sections,  # e.g. {'warnings': '...', 'dosage_and_administration': '...'}
    }

In [21]:
context = get_drug_context('DRUG_GENERIC:6e49f2b49e93d317', graph)
print(f"Generic: {context['generic_name']}")
print(f"Number of brands: {len(context['brands'])}")
print(f"Sections found: {list(context['sections'].keys())}")

Generic: LIDOCAINE
Number of brands: 208
Sections found: ['active_ingredient', 'do_not_use', 'dosage_and_administration', 'inactive_ingredient', 'indications_and_usage', 'keep_out_of_reach_of_children', 'pregnancy_or_breast_feeding', 'purpose', 'warnings']


In [22]:
def build_prompt(user_query: str, context: dict) -> str:
    """
    Turns a get_drug_context() result into a prompt for Llama-3.
    """
    generic_name = context['generic_name']
    brands = context['brands']
    sections = context['sections']

    # Show a capped sample of brand names — 208 brands would eat context
    # budget for no benefit; the model just needs to know brand names exist
    brand_sample = ', '.join(brands[:15])
    brand_note = f" (and {len(brands) - 15} more)" if len(brands) > 15 else ""

    section_order = [
        'indications_and_usage', 'active_ingredient', 'purpose',
        'dosage_and_administration', 'warnings', 'do_not_use',
        'keep_out_of_reach_of_children', 'pregnancy_or_breast_feeding',
        'inactive_ingredient',
    ]
    section_text = "\n\n".join(
        f"## {sec.replace('_', ' ').title()}\n{sections[sec]}"
        for sec in section_order
        if sec in sections and sections[sec]
    )

    prompt = f"""You are answering a question using FDA drug label data.
Answer ONLY using the information below. If the answer isn't in the provided
information, say so clearly instead of guessing.

# Drug: {generic_name}
Brand names include: {brand_sample}{brand_note}

{section_text}

# Question
{user_query}

# Answer
"""
    return prompt

In [23]:
import requests

def ask_llama(prompt: str, model="llama3", temperature=0.1) -> str:
    resp = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": temperature,
                "num_ctx": 8192,  # raised from Ollama's default, since section
                                  # text + brand list can run long
            }
        },
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["response"]

In [24]:
context = get_drug_context('DRUG_GENERIC:6e49f2b49e93d317', graph)
prompt = build_prompt("What should I avoid if I'm pregnant and using lidocaine?", context)
print(prompt)  # sanity check the assembled context before spending an LLM call

answer = ask_llama(prompt)
print(answer)

You are answering a question using FDA drug label data.
Answer ONLY using the information below. If the answer isn't in the provided
information, say so clearly instead of guessing.

# Drug: LIDOCAINE
Brand names include: Sonacaine Topical Anesthetic With Menthol, Salonpas Pain Relieving LIDOCAINE 4% FLEX, DYNAMO DELAY, Lidocaine 5%, EPIC Burn Cream, Natural Seal Itch Relief for Kids, Aspercreme, Nupharmisto, Assured Bikini Smooth Cream, RECTICARE, Tylenol Precise Pain Relieving, SWISS NAVY Endurance Rx, LMR Plus, Hemorrhoid Pain Relief, Curist Lidocaine Patch Maximum Strength (and 193 more)

## Indications And Usage
USES: For temporary relief of pain

## Active Ingredient
ACTIVE INGREDIENTS: Lidocaine 4.00% Menthol 1.00% Topical Anesthetic External Analgesic

## Purpose
Topical Anesthetic External Analgesic

## Dosage And Administration
DIRECTIONS (Adults and Children Over 12 Years): Clean and dry affected area. Remove patch from backing and apply to affected area. Use only one patch 

# Installing FAISS

In [25]:
!pip install faiss-cpu --quiet

# Creating Embeddings

In [26]:
import faiss
import numpy as np
import requests
import pickle
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

OLLAMA_URL = "http://localhost:11434/api/embeddings"
EMBED_MODEL = "nomic-embed-text"
CHECKPOINT_PATH = "embeddings_checkpoint.pkl"
MAX_WORKERS = 8

def node_to_text(node_id: str, data: dict) -> str:
    node_type = data.get("type", "")
    name = (data.get("name") or "").strip()
    description = (data.get("description") or "").strip()

    if node_type in ("drug_generic", "drug_brand"):
        text = name
    elif description:
        text = description
    else:
        text = name

    return f"[{node_type}] {text}"

def get_embedding(text: str):
    try:
        resp = requests.post(OLLAMA_URL, json={"model": EMBED_MODEL, "prompt": text}, timeout=60)
        resp.raise_for_status()
        return np.array(resp.json()["embedding"], dtype="float32")
    except Exception as e:
        print(f"Embedding failed for text[:50]={text[:50]!r}: {e}")
        return None

if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "rb") as f:
        done = pickle.load(f)
    print(f"Resuming from checkpoint: {len(done)} nodes already embedded")
else:
    done = {}

all_nodes = list(graph.nodes(data=True))
todo = [(nid, data) for nid, data in all_nodes if nid not in done]
print(f"Total nodes: {len(all_nodes)}, remaining to embed: {len(todo)}")

CHECKPOINT_EVERY = 2000
processed_since_checkpoint = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_id = {executor.submit(get_embedding, node_to_text(nid, data)): nid for nid, data in todo}
    for future in tqdm(as_completed(future_to_id), total=len(future_to_id), desc="Embedding nodes"):
        nid = future_to_id[future]
        emb = future.result()
        if emb is not None:
            done[nid] = emb
        processed_since_checkpoint += 1
        if processed_since_checkpoint >= CHECKPOINT_EVERY:
            with open(CHECKPOINT_PATH, "wb") as f:
                pickle.dump(done, f)
            processed_since_checkpoint = 0

with open(CHECKPOINT_PATH, "wb") as f:
    pickle.dump(done, f)

print(f"Successfully embedded {len(done)}/{len(all_nodes)} nodes")
failed = [nid for nid, _ in all_nodes if nid not in done]
if failed:
    print(f"Failed nodes ({len(failed)}): {failed[:10]}...")

node_ids = [nid for nid, _ in all_nodes if nid in done]
embeddings = np.vstack([done[nid] for nid in node_ids])
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")

faiss.write_index(index, "graph_nodes.index")
with open("node_id_map.pkl", "wb") as f:
    pickle.dump(node_ids, f)

for nid in node_ids:
    graph.nodes[nid]["embedding"] = done[nid]
with open("graph_with_embeddings.pkl", "wb") as f:
    pickle.dump(graph, f)

Resuming from checkpoint: 0 nodes already embedded
Total nodes: 110843, remaining to embed: 110843


Embedding nodes:  38%|███▊      | 42635/110843 [41:20<1:15:02, 15.15it/s]

Embedding failed for text[:50]='[dosage_and_administration_table] <table width="10': 500 Server Error: Internal Server Error for url: http://localhost:11434/api/embeddings


Embedding nodes:  39%|███▉      | 43173/110843 [41:51<1:10:02, 16.10it/s]

Embedding failed for text[:50]='[dosage_and_administration] 2 DOSAGE AND ADMINISTR': 500 Server Error: Internal Server Error for url: http://localhost:11434/api/embeddings
Embedding failed for text[:50]='[dosage_and_administration_table] <table width="65': 500 Server Error: Internal Server Error for url: http://localhost:11434/api/embeddings


...

Embedding failed for text[:50]='[dosage_and_administration] 2 DOSAGE AND ADMINISTR': 500 Server Error: Internal Server Error for url: http://localhost:11434/api/embeddings
Embedding failed for text[:50]='[dosage_and_administration] 2 DOSAGE AND ADMINISTR': 500 Server Error: Internal Server Error for url: http://localhost:11434/api/embeddings


Embedding nodes: 100%|█████████▉| 110832/110843 [1:47:29<00:00, 13.96it/s]

Embedding failed for text[:50]='[warnings] WARNINGS Potentially Fatal Reactions to': 500 Server Error: Internal Server Error for url: http://localhost:11434/api/embeddings


Embedding nodes: 100%|██████████| 110843/110843 [1:47:29<00:00, 17.18it/s]


Successfully embedded 109960/110843 nodes
Failed nodes (883): ['SECTION:4765d92bd89f44ed', 'SECTION:866eb2f7cc0e69d3', 'SECTION:38de0626d0f28ac4', 'SECTION:98502e2af71f6bb1', 'SECTION:9b863f7069c561d7', 'SECTION:87870a181918c447', 'SECTION:3a069d29499c5484', 'SECTION:98ec139d6e64dfbd', 'SECTION:8944a30843c6d3de', 'SECTION:d2a56c17f6bc40ec']...
FAISS index built: 109960 vectors, dim=768


In [27]:
import faiss
import pickle
import numpy as np

index = faiss.read_index("graph_nodes.index")
with open("node_id_map.pkl", "rb") as f:
    node_ids = pickle.load(f)

print(f"Index size: {index.ntotal} vectors")
print(f"node_id_map size: {len(node_ids)}")
# these two numbers should match exactly — if not, something got
# dropped between embedding and index-building

Index size: 109960 vectors
node_id_map size: 109960


In [28]:
def search_graph(query: str, k: int = 5):
    query_emb = get_embedding(query)
    query_emb = query_emb.reshape(1, -1)
    faiss.normalize_L2(query_emb)

    scores, indices = index.search(query_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        matched_id = node_ids[idx]
        results.append((matched_id, float(score), graph.nodes[matched_id].get('type'), graph.nodes[matched_id].get('name')))
    return results

In [29]:
test_queries = [
    "lidocaine",
    "what should I avoid if pregnant and using lidocaine",
    "aspercreme",
    "is aspercreme safe for children",
    "acetaminophen dosage",
]

for q in test_queries:
    print(f"\nQuery: {q!r}")
    for node_id, score, node_type, name in search_graph(q, k=5):
        print(f"  {score:.3f}  [{node_type}]  {name}")


Query: 'lidocaine'
  0.847  [drug_brand]  Lidocaine
  0.839  [drug_generic]  LIDOCAINE
  0.825  [active_ingredient]  active_ingredient (LIDOCAINE 20%)
  0.821  [drug_generic]  LIDOCANE
  0.810  [drug_brand]  Lidocan

Query: 'what should I avoid if pregnant and using lidocaine'
  0.816  [warnings]  warnings (NUMBING CREAM)
  0.816  [warnings]  warnings (LIDOCAINE NUMBING CREAM)
  0.774  [warnings]  warnings (LIDICAINE)
  0.744  [pregnancy_or_breast_feeding]  pregnancy_or_breast_feeding (CHOLINE SALICYLATE,GUAIFENESIN)
  0.744  [pregnancy_or_breast_feeding]  pregnancy_or_breast_feeding (CHOLINE SALICYLATE, DEXTROMETHORPHAN HYDROBROMIDE, GUAIFENESIN)

Query: 'aspercreme'
  0.835  [drug_brand]  Aspercreme
  0.757  [drug_brand]  Aspercreme Pain Relieving
  0.748  [drug_brand]  Aspercreme Arthritis
  0.741  [drug_brand]  Aspercreme with Lidocaine Dry
  0.741  [drug_brand]  Aspercreme with Lidocaine XL

Query: 'is aspercreme safe for children'
  0.793  [drug_brand]  Aspercreme
  0.740  [drug

#saving files to Drive

In [30]:
import os
import shutil

# --- Create the target folder ---
target_dir = "/content/drive/MyDrive/Research_kgTxagent/graph_rag"
os.makedirs(target_dir, exist_ok=True)

# --- Files created during this session ---
files_to_copy = [
    "graph_nodes.index",           # FAISS index
    "node_id_map.pkl",             # FAISS index -> node ID mapping
    "embeddings_checkpoint.pkl",   # raw embedding checkpoint
    "graph_with_embeddings.pkl",   # original graph + embeddings attached
]

copied = []
missing = []
for fname in files_to_copy:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join(target_dir, fname))
        copied.append(fname)
    else:
        missing.append(fname)

print(f"Copied {len(copied)} files: {copied}")
if missing:
    print(f"Not found in current directory (skipped): {missing}")

print(f"\nContents of {target_dir}:")
for f in os.listdir(target_dir):
    full_path = os.path.join(target_dir, f)
    size_mb = os.path.getsize(full_path) / (1024 * 1024)
    print(f"  {f}  ({size_mb:.1f} MB)")

Copied 4 files: ['graph_nodes.index', 'node_id_map.pkl', 'embeddings_checkpoint.pkl', 'graph_with_embeddings.pkl']

Contents of /content/drive/MyDrive/Research_kgTxagent/graph_rag:
  graph_nodes.index  (322.1 MB)
  node_id_map.pkl  (3.0 MB)
  embeddings_checkpoint.pkl  (328.7 MB)
  graph_with_embeddings.pkl  (415.6 MB)


# Testing: ask your question here!

In [37]:
"""
GraphRAG pipeline — single-cell setup for Google Colab.

Paste this entire file into one Colab cell and run it. On a fresh Colab
runtime, Ollama and the pulled models do NOT persist across sessions
(Colab's local disk is ephemeral), so the install/pull steps below must
run every time you start a new runtime — even though the graph, FAISS
index, and this script itself ARE persisted in Drive.

After this cell finishes, call:
    answer_question("what are the warnings for lidocaine")
"""

import subprocess
import time
import pickle

import requests
import numpy as np
import faiss

# ============================================================
# 1. Install + start Ollama, pull both models
#    (must re-run every fresh Colab runtime — not persisted by Drive)
# ============================================================
print("Installing Ollama...")
subprocess.run("apt-get update -qq && apt-get install -y -qq zstd", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

print("Starting Ollama server...")
subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)
time.sleep(5)

print("Pulling llama3...")
subprocess.run("ollama pull llama3", shell=True)

print("Pulling nomic-embed-text...")
subprocess.run("ollama pull nomic-embed-text", shell=True)

subprocess.run("ollama list", shell=True)

# ============================================================
# 2. Mount Drive (skips silently if not in Colab)
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab — skipping drive mount. "
          "Adjust GRAPH_RAG_DIR below to your local path.")

GRAPH_RAG_DIR = "/content/drive/MyDrive/Research_kgTxagent/graph_rag"

# ============================================================
# 3. Load graph, FAISS index, node ID mapping
# ============================================================
print("Loading graph and FAISS index...")

with open(f"{GRAPH_RAG_DIR}/graph_with_embeddings.pkl", "rb") as f:
    graph = pickle.load(f)

index = faiss.read_index(f"{GRAPH_RAG_DIR}/graph_nodes.index")

with open(f"{GRAPH_RAG_DIR}/node_id_map.pkl", "rb") as f:
    node_ids = pickle.load(f)

print(f"Loaded graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print(f"Loaded FAISS index: {index.ntotal} vectors")
assert index.ntotal == len(node_ids), "Index size and node_id_map size don't match — investigate before proceeding"

# ============================================================
# 4. Pipeline functions
# ============================================================
OLLAMA_EMBED_URL = "http://localhost:11434/api/embeddings"
OLLAMA_GENERATE_URL = "http://localhost:11434/api/generate"
EMBED_MODEL = "nomic-embed-text"
GENERATE_MODEL = "llama3"


def get_embedding(text: str) -> np.ndarray:
    resp = requests.post(OLLAMA_EMBED_URL, json={"model": EMBED_MODEL, "prompt": text}, timeout=60)
    resp.raise_for_status()
    return np.array(resp.json()["embedding"], dtype="float32")


def search_graph(query: str, k: int = 5):
    """Embed a query, search FAISS, return (node_id, score, type, name) tuples."""
    query_emb = get_embedding(query).reshape(1, -1)
    faiss.normalize_L2(query_emb)
    scores, indices = index.search(query_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        matched_id = node_ids[idx]
        results.append((
            matched_id,
            float(score),
            graph.nodes[matched_id].get('type'),
            graph.nodes[matched_id].get('name'),
        ))
    return results


def find_drug_entity(query: str, k: int = 15):
    """
    Search broadly but only trust drug_generic/drug_brand matches for
    entity resolution — section-text matches drift on natural-language
    queries (e.g. symptom/phrase-heavy questions).
    """
    results = search_graph(query, k=k)
    entity_results = [r for r in results if r[2] in ('drug_generic', 'drug_brand')]
    if entity_results:
        return entity_results[0]
    return results[0] if results else None


def get_drug_context(matched_node_id: str, graph) -> dict:
    """Resolve any matched node (drug or section) to its full drug label."""
    node_data = graph.nodes[matched_node_id]
    node_type = node_data.get('type')

    if node_type == 'drug_generic':
        generic_id = matched_node_id
    else:
        generic_candidates = [
            n for n in graph.neighbors(matched_node_id)
            if graph.nodes[n].get('type') == 'drug_generic'
        ]
        if not generic_candidates:
            raise ValueError(f"No drug_generic found for node {matched_node_id} (type={node_type})")
        generic_id = generic_candidates[0]

    generic_data = graph.nodes[generic_id]
    sections = {}
    brands = []
    for neighbor_id in graph.neighbors(generic_id):
        n_data = graph.nodes[neighbor_id]
        n_type = n_data.get('type')
        if n_type == 'drug_brand':
            brands.append(n_data.get('name'))
        elif n_type != 'drug_generic':
            sections[n_type] = n_data.get('description') or n_data.get('name', '')

    return {
        'generic_id': generic_id,
        'generic_name': generic_data.get('name'),
        'brands': brands,
        'sections': sections,
    }


def build_prompt(user_query: str, context: dict) -> str:
    generic_name = context['generic_name']
    brands = context['brands']
    sections = context['sections']

    brand_sample = ', '.join(brands[:15])
    brand_note = f" (and {len(brands) - 15} more)" if len(brands) > 15 else ""

    section_order = [
        'indications_and_usage', 'active_ingredient', 'purpose',
        'dosage_and_administration', 'warnings', 'do_not_use',
        'keep_out_of_reach_of_children', 'pregnancy_or_breast_feeding',
        'inactive_ingredient',
    ]
    section_text = "\n\n".join(
        f"## {sec.replace('_', ' ').title()}\n{sections[sec]}"
        for sec in section_order
        if sections.get(sec)
    )

    return f"""You are answering a question using FDA drug label data.
Answer ONLY using the information below. If the answer isn't in the provided
information, say so clearly instead of guessing.

# Drug: {generic_name}
Brand names include: {brand_sample}{brand_note}

{section_text}

# Question
{user_query}

# Answer
"""


def ask_llama(prompt: str, model: str = GENERATE_MODEL, temperature: float = 0.1) -> str:
    resp = requests.post(
        OLLAMA_GENERATE_URL,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": temperature, "num_ctx": 8192},
        },
        timeout=120,
    )
    resp.raise_for_status()
    return resp.json()["response"]


def answer_question(query: str) -> str:
    """Full pipeline entry point: raw question in, grounded answer out."""
    match = find_drug_entity(query)
    if not match:
        return "No relevant information found."
    top_match_id = match[0]
    context = get_drug_context(top_match_id, graph)
    prompt = build_prompt(query, context)
    return ask_llama(prompt)


print("\nPipeline ready.")
print("Example: answer_question(\"what are the warnings for lidocaine\")")

question = input("Enter your question: ")

answer = answer_question(question)

print(answer)

Installing Ollama...
Starting Ollama server...
Pulling llama3...
Pulling nomic-embed-text...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading graph and FAISS index...
Loaded graph: 110843 nodes, 355909 edges
Loaded FAISS index: 109960 vectors

Pipeline ready.
Example: answer_question("what are the warnings for lidocaine")
Enter your question: "what are the treatments for cold"
According to the FDA drug label data, the treatment for cold sores/fever blisters on the face or lips is:

* Apply a small amount of this product (Camphor, Menthol, Phenol, White Petrolatum) to the affected area 1 to 3 times daily
* Rub in gently

Note that this information only refers to the treatment of cold sores/fever blisters and does not provide information on treatments for general colds.


In [40]:
question = input("Enter your question: ")

answer = answer_question(question)

print(answer)

Enter your question: "What can I take for headaches while pregnant?"
I'm happy to help! However, based on the provided FDA drug label data, there is no specific information about taking Ibuprofen or Acetaminophen for headaches during pregnancy. In fact, the label warns that Ibuprofen should not be used at 20 weeks or later in pregnancy unless specifically directed by a doctor.

It's always best to consult with a healthcare professional for personalized advice on managing headaches and other symptoms during pregnancy. They can help you determine the appropriate course of treatment based on your individual situation and medical history.


In [41]:

question = input("Enter your question: ")

answer = answer_question(question)

print(answer)

Enter your question: What can I take for headaches while pregnant?
According to the FDA drug label data, it is recommended to "ask a health professional before use" if you are pregnant. This suggests that the product should not be used without consulting a healthcare provider while pregnant. Therefore, the answer would be: Consult a healthcare provider before using this product for headaches while pregnant.


## Multi-Node Retrieval Enhancement

The retrieval pipeline was modified to use **multiple relevant nodes** from the knowledge graph instead of restricting retrieval to a single drug. Previously, the system selected the highest-ranked `drug_generic` or `drug_brand` node returned by FAISS, retrieved that drug's complete FDA label, and generated an answer solely from that context. This approach worked well for drug-specific questions but was limited for broader queries involving multiple concepts.

The updated implementation introduces the following changes:

- **`find_drug_entity()`** now returns all relevant FAISS matches above a similarity threshold instead of a single drug entity.
- **`get_drug_context()`** resolves every retrieved node to its corresponding `drug_generic`, removes duplicate drugs, and collects the FDA label sections for each unique drug.
- **`build_prompt()`** combines the retrieved FDA label information from all matched drugs into a single prompt while preserving section organization.
- **`answer_question()`** uses the aggregated contexts to generate a grounded response from multiple FDA labels rather than relying on a single retrieved drug.

This enhancement enables the system to answer broader questions by synthesizing information from multiple relevant FDA drug labels while remaining grounded in the retrieved evidence.

In [42]:
def find_drug_entity(query: str, k: int = 20, threshold: float = 0.70):
    """
    Search the graph and return ALL relevant nodes instead of a single drug.

    Returns:
        List of tuples:
        (node_id, score, node_type, node_name)
    """

    results = search_graph(query, k=k)

    if not results:
        return []

    # Keep only reasonably similar results
    filtered = [r for r in results if r[1] >= threshold]

    # If nothing passes threshold, fall back to top-k
    if not filtered:
        filtered = results

    return filtered

In [43]:
def get_drug_context(matched_nodes, graph):
    """
    Given multiple FAISS matches, resolve each one to its parent drug_generic
    and return the contexts for ALL unique drugs.

    Parameters
    ----------
    matched_nodes : list
        Output of find_drug_entity()

    Returns
    -------
    List of dictionaries.
    """

    contexts = []

    processed_generics = set()

    for matched_node_id, score, node_type, node_name in matched_nodes:

        node_data = graph.nodes[matched_node_id]
        node_type = node_data.get("type")

        # ---------------------------------------------------
        # Resolve to generic drug
        # ---------------------------------------------------
        if node_type == "drug_generic":

            generic_id = matched_node_id

        else:

            generic_candidates = [

                n for n in graph.neighbors(matched_node_id)

                if graph.nodes[n].get("type") == "drug_generic"

            ]

            if not generic_candidates:
                continue

            generic_id = generic_candidates[0]

        # ---------------------------------------------------
        # Skip duplicate drugs
        # ---------------------------------------------------
        if generic_id in processed_generics:
            continue

        processed_generics.add(generic_id)

        generic_data = graph.nodes[generic_id]

        brands = []

        sections = {}

        # ---------------------------------------------------
        # Collect all neighboring sections
        # ---------------------------------------------------
        for neighbor_id in graph.neighbors(generic_id):

            n_data = graph.nodes[neighbor_id]

            n_type = n_data.get("type")

            if n_type == "drug_brand":

                brands.append(n_data.get("name"))

            elif n_type != "drug_generic":

                text = n_data.get("description") or n_data.get("name", "")

                if text:
                    sections[n_type] = text

        contexts.append(

            {

                "generic_id": generic_id,

                "generic_name": generic_data.get("name"),

                "score": score,

                "brands": brands,

                "sections": sections,

            }

        )

    # -------------------------------------------------------
    # Sort by similarity score
    # -------------------------------------------------------
    contexts.sort(key=lambda x: x["score"], reverse=True)

    return contexts

In [44]:
def build_prompt(user_query: str, contexts: list) -> str:
    """
    Builds a prompt using ALL retrieved FDA drug contexts.
    """

    prompt = """You are answering a question using FDA drug label data.

Answer ONLY using the FDA information provided below.

Rules:
- Use ONLY the information provided.
- If multiple drugs are retrieved, compare them when appropriate.
- If the answer cannot be determined from the provided FDA information,
  clearly state that.
- Do not make medical recommendations beyond what the FDA labels state.
- When multiple facts exist, return them as numbered points.

====================================================
FDA LABEL INFORMATION
====================================================

"""

    section_order = [
        "indications_and_usage",
        "active_ingredient",
        "purpose",
        "dosage_and_administration",
        "warnings",
        "do_not_use",
        "keep_out_of_reach_of_children",
        "pregnancy_or_breast_feeding",
        "inactive_ingredient",
    ]

    for context in contexts:

        generic_name = context["generic_name"]
        brands = context["brands"]
        sections = context["sections"]

        brand_sample = ", ".join(brands[:15])

        if len(brands) > 15:
            brand_sample += f" (and {len(brands)-15} more)"

        prompt += f"\n==============================\n"
        prompt += f"Drug: {generic_name}\n"

        if brands:
            prompt += f"Brands: {brand_sample}\n"

        prompt += "------------------------------\n"

        for sec in section_order:

            if sec in sections and sections[sec]:

                title = sec.replace("_", " ").title()

                prompt += f"\n## {title}\n"

                prompt += f"{sections[sec]}\n"

    prompt += f"""

====================================================
QUESTION
====================================================

{user_query}

====================================================
ANSWER
====================================================
"""

    return prompt

In [45]:
def answer_question(query: str) -> str:
    """
    Full GraphRAG pipeline.
    """

    # Retrieve multiple relevant nodes
    matches = find_drug_entity(query)

    if not matches:
        return "No relevant FDA information found."

    # Build contexts for all retrieved drugs
    contexts = get_drug_context(matches, graph)

    if not contexts:
        return "No FDA drug context could be retrieved."

    # Build prompt
    prompt = build_prompt(query, contexts)

    # Uncomment for debugging
    # print(prompt)

    # Ask Llama
    answer = ask_llama(prompt)

    return answer

# testing

In [48]:
matches = find_drug_entity('What can I take for headaches while pregnant?')

In [49]:
print("\n========== FAISS Retrieved Nodes ==========")

for i, (node_id, score, node_type, node_name) in enumerate(matches, 1):
    print(f"{i}.")
    print(f"   Node ID   : {node_id}")
    print(f"   Score     : {score:.4f}")
    print(f"   Type      : {node_type}")
    print(f"   Name      : {node_name}")
    print()


========== FAISS Retrieved Nodes ==========
1.
   Node ID   : SECTION:77491f6c95ffc568
   Score     : 0.7181
   Type      : warnings
   Name      : warnings (SANGUINARIA CANADENSIS ROOT)

2.
   Node ID   : SECTION:c5eb6bbcd44b486f
   Score     : 0.7181
   Type      : warnings
   Name      : warnings (CYCLAMEN PURPURASCENS TUBER, BLACK COHOSH, IRIS VERSICOLOR WHOLE, GELSEMIUM SEMPERVIRENS WHOLE, SANGUINARIA CANADENSIS ROOT)

3.
   Node ID   : SECTION:5fdcc0dd801f0b4b
   Score     : 0.7181
   Type      : warnings
   Name      : warnings (CYCLAMEN PURPURASCENS TUBER)

4.
   Node ID   : SECTION:388464ff44d85167
   Score     : 0.7181
   Type      : warnings
   Name      : warnings (BLACK COHOSH)

5.
   Node ID   : SECTION:7ff649e9ca8b0336
   Score     : 0.7050
   Type      : pregnancy_or_breast_feeding
   Name      : pregnancy_or_breast_feeding (ETHYL ALCOHOL, ISOPROPYL ALCOHOL, BACITRACIN ZINC, NEOMYCIN SULFATE, POLYMYXIN-B SULFATE, DIPHENHYDRAMINE HYDROCHLORIDE, HYDROCORTISONE, IBUPROFEN

In [52]:
question = "What can I take for headaches while pregnant?"

matches = find_drug_entity(question, k=20)

print("\n========== RETRIEVED NODES ==========")

for m in matches:
    print(
        m[1],
        "|",
        m[2],
        "|",
        m[3]
    )


contexts = get_drug_context(matches, graph)

print("\n========== RESOLVED DRUGS ==========")

for c in contexts:
    print("\nDrug:", c["generic_name"])
    print("Score:", c["score"])
    print("Sections:", list(c["sections"].keys()))


========== RETRIEVED NODES ==========
0.7180848121643066 | warnings | warnings (SANGUINARIA CANADENSIS ROOT)
0.7180848121643066 | warnings | warnings (CYCLAMEN PURPURASCENS TUBER, BLACK COHOSH, IRIS VERSICOLOR WHOLE, GELSEMIUM SEMPERVIRENS WHOLE, SANGUINARIA CANADENSIS ROOT)
0.7180848121643066 | warnings | warnings (CYCLAMEN PURPURASCENS TUBER)
0.7180848121643066 | warnings | warnings (BLACK COHOSH)
0.7049801349639893 | pregnancy_or_breast_feeding | pregnancy_or_breast_feeding (ETHYL ALCOHOL, ISOPROPYL ALCOHOL, BACITRACIN ZINC, NEOMYCIN SULFATE, POLYMYXIN-B SULFATE, DIPHENHYDRAMINE HYDROCHLORIDE, HYDROCORTISONE, IBUPROFEN, ACETAMINOPHEN, WATER)
0.7049801349639893 | pregnancy_or_breast_feeding | pregnancy_or_breast_feeding (ETHYL ALCOHOL, ISOPROPYL ALCOHOL, BACITRACIN ZINC, NEOMYCIN SULFATE, POLYMYXIN-B SULFATE)
0.7049801349639893 | pregnancy_or_breast_feeding | pregnancy_or_breast_feeding (ETHYL ALCOHOL, ISOPROPYL ALCOHOL)

========== RESOLVED DRUGS ==========

Drug: SANGUINARIA CANAD

In [50]:
search_terms = [
    "headache",
    "head pain",
    "migraine",
    "analgesic",
    "pain relief"
]

print("\n========== HEADACHE SEARCH ==========")

for term in search_terms:
    print("\nQuery:", term)

    results = find_drug_entity(term)

    for r in results[:5]:
        print(
            r[2],
            "|",
            r[3],
            "| score:",
            r[1]
        )


========== HEADACHE SEARCH ==========

Query: headache
purpose | purpose (BELLADONNA, GLONOINUM, COFFEA, NATRUM CARB, NATRUM MUR.) | score: 0.7945243716239929
drug_brand | Headache | score: 0.7843437194824219
purpose | purpose (ATROPA BELLADONNA, GELSEMIUM SEMPERVIRENS ROOT, IPECAC AND IRIS VERSICOLOR ROOT) | score: 0.7773514986038208
drug_brand | Headache-Migraine | score: 0.7533246278762817
drug_brand | HEADACHE HP | score: 0.7516359090805054

Query: head pain
drug_brand | Aching Head Rub | score: 0.7309045791625977
purpose | purpose (MENTHOL, UNSPECIFIED FORM, ATROPA BELLADONNA, IRIS VERSICOLOR ROOT, SANGUINARIA CANADENSIS ROOT) | score: 0.7065355777740479

Query: migraine
purpose | purpose (BELLADONNA, GLONOINUM, COFFEA, NATRUM CARB, NATRUM MUR.) | score: 0.7867599725723267
purpose | purpose (BLACK COHOSH, IRIS VERSICOLOR ROOT, SANGUINARIA CANADENSIS ROOT, GELSEMIUM SEMPERVIRENS ROOT, AND SPIGELIA ANTHELMIA) | score: 0.7783843278884888
purpose | purpose (GELSEMIUM SEMPERVIRENS ROO

In [51]:
query = "headache during pregnancy"

results = find_drug_entity(query)

print("\n========== PREGNANCY HEADACHE ==========")

for r in results[:20]:
    print(
        r[2],
        "|",
        r[3],
        "|",
        r[1]
    )


========== PREGNANCY HEADACHE ==========
pregnancy_or_breast_feeding | pregnancy_or_breast_feeding (BRYONIA ALBA ROOT, EUPHRASIA STRICTA, CALCIUM SULFIDE, SODIUM CHLORIDE, PHOSPHORUS, ANEMONE PATENS, RUMEX CRISPUS ROOT, AND SILICON DIOXIDE) | 0.6701008081436157
pregnancy_or_breast_feeding | pregnancy_or_breast_feeding (ETHYL ALCOHOL, ISOPROPYL ALCOHOL, BACITRACIN ZINC, NEOMYCIN SULFATE, POLYMYXIN-B SULFATE, DIPHENHYDRAMINE HYDROCHLORIDE, HYDROCORTISONE, IBUPROFEN, ACETAMINOPHEN, WATER) | 0.6678337454795837
pregnancy_or_breast_feeding | pregnancy_or_breast_feeding (ETHYL ALCOHOL, ISOPROPYL ALCOHOL, BACITRACIN ZINC, NEOMYCIN SULFATE, POLYMYXIN-B SULFATE) | 0.6678337454795837
pregnancy_or_breast_feeding | pregnancy_or_breast_feeding (ETHYL ALCOHOL, ISOPROPYL ALCOHOL) | 0.6678337454795837
pregnancy_or_breast_feeding | pregnancy_or_breast_feeding (LIDOCAINE AND MENTHOL) | 0.6554924249649048
pregnancy_or_breast_feeding | pregnancy_or_breast_feeding (IBUPROFEN, DIPHENHYDRAMINE CITRATE) | 0.6

In [53]:
keywords = ["headache", "pregnancy"]

matches = []

for node_id, data in graph.nodes(data=True):

    text = (
        str(data.get("name", "")) +
        " " +
        str(data.get("description", ""))
    ).lower()

    if all(k in text for k in keywords):
        matches.append(
            (
                node_id,
                data.get("type"),
                data.get("name")
            )
        )

print("Nodes containing both headache and pregnancy:")
print("Count:", len(matches))

for m in matches[:20]:
    print(m)

Nodes containing both headache and pregnancy:
Count: 71
('SECTION:326f65a0b5d25544', 'warnings', 'warnings (ACETAMINOPHEN 325MG/ DEXTROMETHORPHAN HBR 10MG/ PHENYLEPHRINE HCL 5MG)')
('SECTION:4d81f7b620870aac', 'warnings', 'warnings (ACETAMINOPHEN ASPIRIN CAFFEINE)')
('SECTION:6434cd19204ba8aa', 'warnings', 'warnings (ACETAMINOPHEN, ASPIRIN, CAFFEINE)')
('SECTION:d2ebbf10f3f9de06', 'warnings', 'warnings (ACETAMINOPHEN-ASPIRIN-CAFFEINE)')
('SECTION:bbec9361ce6442b4', 'warnings', 'warnings (ASPIRIN, CAFFEINE)')
('SECTION:a4ac6410bfa5e548', 'warnings', 'warnings (ASPIRIN, CHLORPHENIRAMINE MALEATE, AND PHENYLEPHRINE BITARTRATE)')
('SECTION:5231d9b2ef946dad', 'warnings', 'warnings (ASPIRIN, DOXYLAMINE SUCCINATE, DEXTROMETHORPHAN HYDROBROMIDE, PHENYLEPHRINE BITARTRATE)')
('SECTION:6edc4fbfda1ba87a', 'warnings', 'warnings (ASPIRIN,CAFFEINE)')
('SECTION:0edc06508dbe339a', 'warnings', 'warnings (ASPIRIN. CAFFEINE)')
('SECTION:23a3c11f0054f0b7', 'warnings', 'warnings (ASPRIN, DEXTROMETHORPHAN HYD

In [54]:
drugs_with_headache = []
drugs_with_pregnancy = []

for node_id, data in graph.nodes(data=True):

    if data.get("type") != "drug_generic":
        continue

    neighbors = [
        graph.nodes[n].get("type")
        for n in graph.neighbors(node_id)
    ]

    names = [
        str(graph.nodes[n].get("name","")).lower()
        for n in graph.neighbors(node_id)
    ]

    has_headache = any("headache" in x for x in names)
    has_pregnancy = "pregnancy_or_breast_feeding" in neighbors

    if has_headache and has_pregnancy:
        print(
            "Drug:",
            data.get("name")
        )

Drug: NAPROXEN SODIUM
Drug: ACETAMINOPHEN, DEXTROMETHORPHAN HBR, PHENYLEPHRINE HCL
Drug: ACETAMINOPHEN, DEXTROMETHORPHAN HYDROBROMIDE, PHENYLEPHRINE HYDROCHLORIDE
Drug: ASPIRIN AND CAFFEINE
Drug: ACETAMINOPHEN, GUAIFENESIN, PHENYLEPHRINE HCL
Drug: ACETAMINOPHEN AND PHENYLEPHRINE HYDROCHLORIDE
Drug: ACETAMINOPHEN, ASPIRIN, AND CAFFEINE
Drug: ACETAMINOPHEN, GUAIFENESIN, AND PHENYLEPHRINE HYDROCHLORIDE
Drug: ACETAMINOPHEN AND PHENYLEPHRINE HCL
Drug: ACETAMINOPHEN, ASPIRIN, CAFFEINE
Drug: ACETAMINOPHEN, DEXTROMETHORPHAN HYDROBROMIDE, AND PHENYLEPHRINE HYDROCHLORIDE
Drug: ACETAMINOPHEN, PHENYLEPHRINE HCL
Drug: ACETAMINOPHEN, ASPIRIN AND CAFFEINE
Drug: ACETAMINOPHEN, CHLORPHENIRAMINE MALEATE, PHENYLEPHRINE HCL
Drug: ACETAMINOPHEN, DIPHENHYDRAMINE HCL, PHENYLEPHRINE HCL
Drug: CAPSICUM
Drug: ACETAMINOPHEN, DIPHENHYDRAMINE HCL AND PHENYLEPHRINE HCL
Drug: EXTRA STRENGHT HEADACHE RELIEF- ACETAMINOPHEN, ASPIRIN, CAFFEINE TABLET
Drug: ATROPA BELLADONNA - BRYONIA ALBA ROOT - BLACK COHOSH - ARABICA C

# Ask your question:

In [55]:

question = input("Enter your question: ")

answer = answer_question(question)

print("\nAnswer:")
print(answer)

Enter your question: What can I take for headaches while pregnant?

Answer:
According to the FDA drug label data provided, there is no specific information on taking Ibuprofen (200mg) or Acetaminophen (325mg) for headaches while pregnant. However, it is mentioned that:

* For Ibuprofen: "It is especially important not to use ibuprofen at 20 weeks or later in pregnancy unless specifically directed to do so by a doctor because it may cause problems in the unborn child or complications during delivery."
* For Acetaminophen: No specific information on taking Acetaminophen for headaches while pregnant.

Therefore, it is recommended that you consult with your healthcare provider before taking any medication, including Ibuprofen and Acetaminophen, to discuss safe options for managing headaches during pregnancy.
